# Лабораторная работа 15. RL с проверяемым вознаграждением: GRPO (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 5:** дообучение и выравнивание

**Цель.** Реализовать Group Relative Policy Optimization (Shao et al., 2024; DeepSeek-R1) на задаче с автоматически проверяемым ответом и сравнить RL с SFT и self-consistency.

**Что нужно сдать:**
1. SFT символьной mini-GPT на сложении двузначных чисел (стартовая политика).
2. GRPO: групповое преимущество, clipped ratio, KL-штраф к референсной политике.
3. Кривые: вознаграждение, точность, KL, доля групп без сигнала.
4. Эксперимент с reward hacking при неполном верификаторе.
5. Сравнение GRPO, продолженного SFT и self-consistency (majority vote).

**Модель.** В режиме `FAST_DEV_RUN` используется символьная mini-GPT из ЛР 6 (около 0.4 млн параметров), всё обучение занимает несколько минут на CPU. Дополнительное задание для GPU — та же схема для Qwen2.5-0.5B с LoRA на задачах GSM8K (см. конец ноутбука).

**Теория.** Для промпта \(q\) сэмплируется группа \(o_1, \dots, o_G\) из \(\pi_{old}\), вознаграждения \(r_i\) нормируются внутри группы: \(\hat A_i = (r_i - \mathrm{mean}(r))/(\mathrm{std}(r) + \varepsilon)\). Функция критика не нужна. Цель:
\[
\mathcal{J} = \frac1G\sum_i \frac1{|o_i|}\sum_t \Big[\min\big(\rho_{i,t}\hat A_i,\ \mathrm{clip}(\rho_{i,t}, 1-\epsilon, 1+\epsilon)\hat A_i\big) - \beta\,\mathbb{D}_{KL}^{(t)}\Big],\qquad
\rho_{i,t} = \frac{\pi_\theta(o_{i,t}\mid q, o_{i,<t})}{\pi_{old}(o_{i,t}\mid q, o_{i,<t})},
\]
\(\mathbb{D}_{KL}^{(t)} = \frac{\pi_{ref}}{\pi_\theta} - \log\frac{\pi_{ref}}{\pi_\theta} - 1\) — несмещённая неотрицательная оценка KL (k3, Schulman).

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

In [ ]:
# ==== Эталонный mini-GPT (решение ЛР 6), используется как готовый модуль ====
from dataclasses import dataclass, asdict

class CharTokenizer:
    def __init__(self, text):
        self.chars = sorted(set(text)); self.stoi = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
    def encode(self, s): return [self.stoi[c] for c in s if c in self.stoi]
    def decode(self, ids): return "".join(self.chars[i] for i in ids)

@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.0

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def rope_cache(T, d_head, base=10000.0, device="cpu", scale=1.0):
    inv = 1.0 / (base ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(T, device=device).float() / scale
    ang = torch.outer(t, inv)                        # (T, d_head/2)
    return ang.cos(), ang.sin()

def apply_rope(x, cos, sin):                         # x: (B, H, T, d_head)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.flatten(-2)

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head, self.d_head = cfg.n_head, cfg.n_embd // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.d_head).transpose(1, 2) for t in (q, k, v))
        q, k = apply_rope(q, cos[:T], sin[:T]), apply_rope(k, cos[:T], sin[:T])
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False)
    def forward(self, x): return self.w2(F.silu(self.w1(x)) * self.w3(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1, self.attn = RMSNorm(cfg.n_embd), CausalSelfAttention(cfg)
        self.n2, self.mlp = RMSNorm(cfg.n_embd), SwiGLU(cfg.n_embd, int(8 * cfg.n_embd / 3))
    def forward(self, x, cos, sin):
        x = x + self.attn(self.n1(x), cos, sin)
        return x + self.mlp(self.n2(x))

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm = RMSNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.wte.weight                    # weight tying
        cos, sin = rope_cache(4 * cfg.block_size, cfg.n_embd // cfg.n_head)
        self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
        self.apply(self._init)
        for n, p in self.named_parameters():                  # масштабирование выходных проекций по глубине
            if n.endswith("proj.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.n_layer))
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        x = self.wte(idx)
        for b in self.blocks: x = b(x, self.cos, self.sin)
        logits = self.head(self.norm(x))
        loss = None if targets is None else F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1] / temperature, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def get_batch(data, block_size, batch_size, device=DEVICE):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, data, block_size, batch_size=32, iters=20):
    model.eval(); losses = [model(*get_batch(data, block_size, batch_size))[1].item() for _ in range(iters)]
    model.train(); return float(np.mean(losses))

def train_lm(model, train_data, val_data, steps, lr=3e-3, batch_size=32, warmup=50, log_every=100):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))
    hist = []
    for s in range(steps):
        x, y = get_batch(train_data, model.cfg.block_size, batch_size)
        _, loss = model(x, y); opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if s % log_every == 0 or s == steps - 1:
            hist.append((s, loss.item(), estimate_loss(model, val_data, model.cfg.block_size)))
    return hist

def get_reference_gpt(text, cfg=None, steps=None, name=None):
    """Загружает или обучает эталонную символьную модель на тексте."""
    tok = CharTokenizer(text)
    data = torch.tensor(tok.encode(text), dtype=torch.long)
    n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
    cfg = cfg or (GPTConfig(tok.vocab_size, 64, 2, 4, 64) if FAST_DEV_RUN else GPTConfig(tok.vocab_size, 256, 6, 6, 384))
    name = name or f"ref_gpt_L{cfg.n_layer}_d{cfg.n_embd}_T{cfg.block_size}.pt"
    path = ARTIFACTS / name
    model = GPT(cfg).to(DEVICE)
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=DEVICE))
    else:
        train_lm(model, train_data, val_data, steps or (300 if FAST_DEV_RUN else 5000))
        torch.save(model.state_dict(), path)
    return model, tok, train_data, val_data

## 0. Задача и токенизатор

Промпт `"47+85="`, ответ `"132\n"`. Перевод строки — конец ответа.

In [ ]:
tok = CharTokenizer("0123456789+=\n")
NL = tok.stoi["\n"]
def make_problem(rng): a, b = rng.randint(10, 99), rng.randint(10, 99); return f"{a}+{b}=", str(a + b)
rng = random.Random(0); sft_data = [make_problem(rng) for _ in range(20_000)]
rng = random.Random(1); test_data = [make_problem(rng) for _ in range(500)]
rl_rng = random.Random(2)
PROMPT_LEN, MAX_NEW = 6, 4

def pad_encode(strings):
    T = max(map(len, strings)); return torch.tensor([tok.encode(s) + [NL] * (T - len(s)) for s in strings])

## 1. SFT — стартовая политика

Обучите mini-GPT на парах «промпт → ответ» с loss только по ответу (метки промпта −100). Остановите обучение рано, чтобы точность была заметно ниже 100%: так RL есть что улучшать.

In [ ]:
cfg = GPTConfig(vocab_size=tok.vocab_size, block_size=16, n_layer=2, n_head=4, n_embd=128)

def sft_step(model, opt, problems):
    x = pad_encode([p + a + "\n" for p, a in problems]).to(DEVICE)
    y = x.clone(); y[:, :PROMPT_LEN] = -100
    for i, (p, a) in enumerate(problems): y[i, PROMPT_LEN + len(a) + 1:] = -100
    _, loss = model(x[:, :-1], y[:, 1:])
    opt.zero_grad(); loss.backward(); opt.step(); return loss.item()

@torch.no_grad()
def generate(model, prompts, n=1, temperature=1.0, greedy=False):
    """Возвращает ids (B*n, PROMPT_LEN + MAX_NEW) и строки ответов."""
    idx = torch.tensor([tok.encode(p) for p in prompts]).repeat_interleave(n, 0).to(DEVICE)
    for _ in range(MAX_NEW):
        logits = model(idx)[0][:, -1] / (1 if greedy else temperature)
        nxt = logits.argmax(-1, keepdim=True) if greedy else torch.multinomial(logits.softmax(-1), 1)
        idx = torch.cat([idx, nxt], 1)
    outs = [tok.decode(r[PROMPT_LEN:].tolist()).split("\n")[0] for r in idx]
    return idx, outs

def accuracy(model, data=test_data):
    _, outs = generate(model, [p for p, _ in data], greedy=True)
    return float(np.mean([o == a for o, (_, a) in zip(outs, data)]))

set_seed(0); policy0 = GPT(cfg).to(DEVICE); opt = torch.optim.AdamW(policy0.parameters(), lr=3e-3)
SFT_STEPS = 2000 if FAST_DEV_RUN else 3000     # подобрано так, чтобы стартовая точность была около 0.3
t0 = time.time(); rng = random.Random(0)
for s in range(SFT_STEPS):
    loss = sft_step(policy0, opt, rng.sample(sft_data, 64))
    if (s + 1) % 500 == 0: print(f"шаг {s + 1}: loss {loss:.3f}, точность {accuracy(policy0):.3f}")
print(f"SFT: {time.time() - t0:.0f} c, параметров {n_params(policy0):,}")
_ = policy0.eval()

## 2. GRPO

Реализуйте:
1. `token_logps(model, ids)` — log-вероятности сгенерированных токенов (позиции `PROMPT_LEN..`), форма `(N, MAX_NEW)`.
2. Маску ответа: токены до первого `\n` включительно.
3. Вознаграждение `reward_fn(outs, answers)`.
4. Шаг GRPO: сэмплирование группы из \(\pi_{old}\), групповые преимущества, `MU` градиентных шагов по clipped-цели с KL-штрафом к \(\pi_{ref}\) (= стартовая SFT-политика).

In [ ]:
import copy
def token_logps(model, ids):
    # TODO: ваш код
    raise NotImplementedError

def response_mask(ids):
    # TODO: ваш код
    raise NotImplementedError

def exact_reward(outs, answers): return torch.tensor([float(o == a) for o, a in zip(outs, answers)])

def grpo_train(reward_fn, steps, B=16, G=8, MU=2, lr=3e-4, clip=0.2, beta=0.04, seed=0, track_true=True, std_norm=True):
    set_seed(seed); pol = copy.deepcopy(policy0); ref = copy.deepcopy(policy0).eval()
    for p in ref.parameters(): p.requires_grad_(False)
    opt = torch.optim.AdamW(pol.parameters(), lr=lr); rng = random.Random(seed); log = []
    for s in range(steps):
        probs = rng.sample(sft_data, B); prompts, answers = [p for p, _ in probs], [a for _, a in probs]
        pol.eval(); ids, outs = generate(pol, prompts, n=G)
        answers_g = [a for a in answers for _ in range(G)]
        # TODO: ваш код
        raise NotImplementedError
        if track_true and (s % 25 == 0 or s == steps - 1): entry["точность"] = accuracy(pol)
        log.append(entry)
    return pol, pd.DataFrame(log)

import pandas as pd
print("стартовая точность (greedy):", accuracy(policy0))
GRPO_STEPS = 200 if FAST_DEV_RUN else 1000
t0 = time.time(); pol_grpo, log_grpo = grpo_train(exact_reward, GRPO_STEPS); print(f"GRPO: {time.time() - t0:.0f} c, точность {accuracy(pol_grpo):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(17, 3.5))
sm = log_grpo.rolling(10, min_periods=1).mean()
ax[0].plot(sm["reward"]); ax[0].set_title("средний reward (сэмплы)")
acc = log_grpo.dropna(subset=["точность"]); ax[1].plot(acc["шаг"], acc["точность"], "o-"); ax[1].set_title("точность (greedy)")
ax[2].plot(sm["KL"]); ax[2].set_title("KL к π_ref (k3)"); ax[3].plot(sm["доля групп без сигнала"]); ax[3].set_title("доля групп с std(r)=0")
for a in ax: a.set_xlabel("шаг")
plt.tight_layout(); plt.show()

## 3. Абляции GRPO

Сравните на половине числа шагов основного запуска: (а) \(\beta = 0\) (без KL), (б) без нормировки на std (`std_norm=False`, только вычитание среднего, как в Dr. GRPO), (в) \(G = 2\) вместо 8 при том же числе сэмплов на шаг (\(B\) увеличить в 4 раза). Достаточно итоговой точности и KL.

In [ ]:
# TODO: ваш код (три абляции и сводная таблица)

## 4. Reward hacking

Неполный верификатор: награда 1, если у ответа правильная **длина** и правильная **первая цифра** (остальные цифры не проверяются). Обучите GRPO с этой наградой, логируя прокси-награду и истинную точность. Объясните наблюдаемое и предложите, как обнаруживать такие ситуации на практике.

In [ ]:
def proxy_reward(outs, answers):
    # TODO: ваш код
    raise NotImplementedError

t0 = time.time(); pol_hack, log_hack = grpo_train(proxy_reward, GRPO_STEPS); print(f"{time.time() - t0:.0f} c")
_, outs = generate(pol_hack, [p for p, _ in test_data[:300]], greedy=True)
print("прокси-награда на тесте:", proxy_reward(outs, [a for _, a in test_data[:300]]).mean().item(), " истинная точность:", accuracy(pol_hack))
for (p, a), o in list(zip(test_data, outs))[:6]: print(f"{p}{o:6s} (верно {a})")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(log_hack["reward"].rolling(10, min_periods=1).mean(), label="прокси-награда")
acc = log_hack.dropna(subset=["точность"]); ax.plot(acc["шаг"], acc["точность"], "o-", label="истинная точность"); ax.legend(); ax.set_xlabel("шаг"); plt.show()

## 5. Сравнение с SFT и self-consistency

1. **Продолженный SFT:** дообучите стартовую политику ещё на \(\text{GRPO\_STEPS}\) шагов SFT по правильным ответам.
2. **Self-consistency:** для стартовой и GRPO-политик сгенерируйте 8 сэмплов на задачу (temperature 1) и возьмите самый частый ответ.
3. **pass@8** для обеих политик.

Сведите таблицу и ответьте: улучшает ли GRPO pass@k или в основном «заостряет» распределение (pass@1 растёт, pass@8 почти нет)?

In [ ]:
from collections import Counter
# TODO: ваш код (продолженный SFT, majority@8 и pass@8 для стартовой и GRPO-политик)

## Дополнительно (GPU): GRPO для Qwen2.5-0.5B на GSM8K

Перенесите `grpo_train` на `Qwen/Qwen2.5-0.5B-Instruct` с LoRA из ЛР 13: промпт с инструкцией писать ответ после `####`, награда — совпадение числа после `####` с эталоном плюс небольшая награда за формат. Используйте \(G = 8\), до 256 новых токенов, \(\beta = 0.04\), lr 1e-5…5e-5. Сравните точность на 200 задачах теста до и после 200 шагов. Обратите внимание на рост длины ответа и на долю групп без сигнала.

## Вопросы

1. Почему GRPO обходится без функции ценности? Чем групповое среднее хуже или лучше обученного критика (смещение, дисперсия, стоимость)?
2. Зачем clipping при \(\mu > 1\) градиентных шагах на одной выборке? Что происходит при \(\mu = 1\)?
3. Почему оценка k3 неотрицательна и несмещена? Зачем KL-штраф, если вознаграждение проверяемое?
4. Что такое «группы без сигнала» и как их доля меняется по ходу обучения? Как с этим борются (DAPO: dynamic sampling, отбор задач по сложности)?
5. Сравните с DPO из ЛР 14: где здесь on-policy данные и почему это важно?